# Sentinel — Phase 2.1 fine-tuning (Colab GPU)

Runs the **real** detector fine-tune on a free Colab GPU — the job that the local CPU can only smoke-test. Trains both D1 bake-off candidates (YOLO12 and RF-DETR), evaluates them, exports the winner, and zips the weights for download back into the Sentinel repo.

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

See `training/README.md` and `DECISIONS.md` D1 for the reasoning behind every choice here.

## 1. Environment

In [ ]:
!pip -q install ultralytics "rfdetr[train,loggers]" roboflow sahi
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Dataset

Two options. Pick one.

**Option A (recommended for the real pilot) — your own Roboflow dataset.** On the dataset's Roboflow Universe page, click *Download this Dataset* → it gives you an exact snippet. Paste it below. Export **YOLOv8** format for YOLO12, and **COCO** format for RF-DETR. The classes should be your remote-home set: `person`, `vehicle`, `package`, `animal`.

**Option B (zero-config fallback) — Pascal VOC.** A real academic dataset that auto-downloads and covers `person`/`car`/`dog`/`cat` (3 of 4 pilot classes; no `package`). Good for a first real run end-to-end. Set `USE_VOC = True`.

In [ ]:
USE_VOC = True  # set False and fill in Option A below for the real pilot dataset

if USE_VOC:
    YOLO_DATA = 'VOC.yaml'   # Ultralytics auto-downloads
    COCO_DIR = None          # RF-DETR step skipped for VOC (needs COCO-format export)
else:
    # --- Option A: paste your Roboflow snippets here ---
    from roboflow import Roboflow
    rf = Roboflow(api_key='PASTE_YOUR_KEY')   # or os.environ['ROBOFLOW_API_KEY']
    # YOLO format for YOLO12:
    yolo_ds = rf.workspace('WORKSPACE').project('PROJECT').version(1).download('yolov8')
    YOLO_DATA = yolo_ds.location + '/data.yaml'
    # COCO format for RF-DETR:
    coco_ds = rf.workspace('WORKSPACE').project('PROJECT').version(1).download('coco')
    COCO_DIR = coco_ds.location
print('YOLO_DATA =', YOLO_DATA)

## 3. Train YOLO12

From COCO-pretrained `yolo12s.pt` (not the bare architecture). Backbone frozen for the first stretch to avoid catastrophic forgetting on a small custom set, per `training/README.md`.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo12s.pt')
model.train(
    data=YOLO_DATA,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    freeze=10,           # freeze backbone early; see training/README.md
    patience=20,         # early-stop when val plateaus
    project='/content/sentinel_runs',
    name='yolo12',
    # CCTV-realistic augmentation on top of defaults:
    hsv_v=0.5, degrees=5, translate=0.1, scale=0.5, fliplr=0.5, mosaic=1.0,
)
print('YOLO12 best weights: /content/sentinel_runs/yolo12/weights/best.pt')

## 4. Train RF-DETR (skip if using VOC)

In [ ]:
if COCO_DIR:
    from rfdetr import RFDETRBase
    m = RFDETRBase()
    m.train(
        dataset_dir=COCO_DIR,
        epochs=60,
        batch_size=8,
        grad_accum_steps=2,
        lr=1e-4,
        output_dir='/content/sentinel_runs/rfdetr',
    )
    print('RF-DETR weights in /content/sentinel_runs/rfdetr/')
else:
    print('Skipped — RF-DETR needs a COCO-format dataset (Option A).')

## 5. Bake-off — held-out mAP

Compare on the same held-out split. The full bake-off (DECISIONS.md D1 / TESTING.md) also weighs false-alarms-per-camera-per-day and edge latency — those need the live footage and the target edge box, so they happen back in the Sentinel repo, not here.

In [ ]:
metrics = YOLO('/content/sentinel_runs/yolo12/weights/best.pt').val(data=YOLO_DATA)
print('YOLO12  mAP50:', round(metrics.box.map50, 4), ' mAP50-95:', round(metrics.box.map, 4))

## 6. Export the winner

ONNX is the portable edge format; TensorRT export happens on the actual Jetson (it needs that hardware). The `.pt` is what `edge/detector.py ModelDetector` loads directly.

In [ ]:
YOLO('/content/sentinel_runs/yolo12/weights/best.pt').export(format='onnx')
print('Exported ONNX next to best.pt')

## 7. Download weights back to the Sentinel repo

Drop the downloaded `best.pt` into `data/models/` in the repo and run:
```
SENTINEL_DETECTOR_BACKEND=model SENTINEL_DETECTOR_MODEL_PATH=./data/models/best.pt python -m edge.main
```

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/sentinel_yolo12_weights', 'zip', '/content/sentinel_runs/yolo12/weights')
files.download('/content/sentinel_yolo12_weights.zip')